# 🚀 Mistral-7B Fine-tuning с QLoRA
## Шаг 2: Fine-tuning + Шаг 3: Evaluation

Этот ноутбук выполняет:
- **Шаг 2**: Дообучение Mistral-7B через QLoRA (4-bit quantization + LoRA адаптеры)
- **Шаг 3**: Оценку базовой и дообученной модели через BERTScore

**Требования к железу**: NVIDIA GPU с минимум 8GB VRAM (RTX 3060 / T4 и выше)

---
## 📦 Ячейка 1: Установка зависимостей

In [ ]:
# Устанавливаем все необходимые библиотеки
# Запускай эту ячейку один раз перед началом работы

!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    datasets==2.21.0 \
    trl==0.10.1 \
    bitsandbytes==0.43.3 \
    accelerate==0.34.2 \
    bert-score==0.3.13 \
    matplotlib==3.9.2 \
    pandas==2.2.2 \
    scipy==1.14.1 \
    sentencepiece==0.2.0 \
    protobuf==5.27.3

print("✅ Все зависимости установлены!")

---
## 🔍 Ячейка 2: Проверка GPU

In [ ]:
# Проверяем наличие и характеристики GPU перед началом работы
# Без GPU запуск этого ноутбука невозможен

import torch
import subprocess

print("=" * 50)
print("🔍 Проверка GPU")
print("=" * 50)

# Проверяем доступность CUDA
if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ CUDA не найдена! Убедитесь, что:\n"
        "1. У вас установлен NVIDIA GPU\n"
        "2. Установлены драйверы NVIDIA\n"
        "3. Установлена версия PyTorch с поддержкой CUDA"
    )

# Выводим информацию о GPU
gpu_count = torch.cuda.device_count()
print(f"✅ CUDA доступна")
print(f"📊 Количество GPU: {gpu_count}")
print()

for i in range(gpu_count):
    gpu_name = torch.cuda.get_device_name(i)
    gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1024**3  # Конвертируем байты в GB
    print(f"GPU {i}: {gpu_name}")
    print(f"  💾 Видеопамять: {gpu_memory:.1f} GB")

print()
print(f"🔧 Версия CUDA: {torch.version.cuda}")
print(f"🔧 Версия PyTorch: {torch.__version__}")

# Устанавливаем основное устройство
DEVICE = "cuda"
print(f"\n✅ Рабочее устройство: {DEVICE}")

---
## 📂 Ячейка 3: Загрузка и подготовка датасета

In [ ]:
# Загружаем датасет из JSONL файла и преобразуем в формат для instruction tuning
# Формат входного файла: {"instruction": "...", "response": "..."}

import json
import random
from datasets import Dataset

# ========== КОНФИГУРАЦИЯ ==========
DATASET_PATH = "cleaned_dataset.jsonl"  # Путь к вашему датасету
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"  # Модель из Hugging Face
ADAPTER_OUTPUT_DIR = "./adapter"  # Папка для сохранения LoRA адаптера
EVAL_SAMPLES = 20  # Количество примеров для оценки (Шаг 3)
RANDOM_SEED = 42  # Фиксируем seed для воспроизводимости
# ==================================

random.seed(RANDOM_SEED)

print("📂 Загружаем датасет...")

# Читаем JSONL файл построчно
raw_data = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # Пропускаем пустые строки
            raw_data.append(json.loads(line))

print(f"✅ Загружено примеров: {len(raw_data)}")
print(f"\n📋 Пример первой записи:")
print(f"  Instruction: {raw_data[0]['instruction'][:100]}...")
print(f"  Response: {raw_data[0]['response'][:100]}...")

In [ ]:
# Преобразуем данные в формат Mistral Instruct
# Mistral-Instruct использует специальный формат: [INST] ... [/INST] ответ

def format_instruction(example):
    """
    Преобразует пример датасета в формат Mistral Instruct.
    
    Mistral Instruct формат:
    <s>[INST] {instruction} [/INST] {response}</s>
    """
    instruction = example["instruction"]
    response = example["response"]
    
    # Формируем текст в формате Mistral
    formatted_text = f"<s>[INST] {instruction} [/INST] {response}</s>"
    
    return {"text": formatted_text}


# Создаём Hugging Face Dataset из списка
dataset = Dataset.from_list(raw_data)

# Применяем форматирование ко всему датасету
dataset = dataset.map(
    format_instruction,
    remove_columns=dataset.column_names  # Удаляем старые колонки
)

# Делим на train/validation (90% / 10%)
split_dataset = dataset.train_test_split(test_size=0.1, seed=RANDOM_SEED)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"✅ Датасет подготовлен:")
print(f"  📚 Train: {len(train_dataset)} примеров")
print(f"  📝 Eval: {len(eval_dataset)} примеров")
print(f"\n📋 Пример отформатированного текста:")
print(train_dataset[0]["text"][:300] + "...")

---
## 🤖 Ячейка 4: Загрузка модели с 4-bit квантизацией (QLoRA)

In [ ]:
# Загружаем Mistral-7B в 4-bit квантизации (QLoRA)
# 
# QLoRA = Quantized LoRA:
# - Основная модель загружается в 4-bit (экономит VRAM)
# - LoRA адаптеры обучаются в float16/bfloat16
# - Результат: обучение 7B модели на GPU с 8-16GB VRAM

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

print("⚙️ Настраиваем 4-bit квантизацию (QLoRA)...")

# Конфигурация 4-bit квантизации через bitsandbytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Загружаем модель в 4-bit
    bnb_4bit_use_double_quant=True,         # Двойная квантизация для экономии памяти
    bnb_4bit_quant_type="nf4",              # NF4 (NormalFloat4) — оптимальный тип квантизации
    bnb_4bit_compute_dtype=torch.bfloat16   # Вычисления в bfloat16 для скорости
)

print("\n🤖 Загружаем токенизатор...")

# Загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Устанавливаем padding token (необходим для батчевого обучения)
# Mistral использует EOS как PAD токен
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Padding справа для causal LM

print(f"  ✅ Токенизатор загружен")
print(f"  📊 Размер словаря: {tokenizer.vocab_size}")

print("\n🤖 Загружаем модель в 4-bit...")
print("   (это может занять 5-10 минут при первой загрузке)")

# Загружаем базовую модель в 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",          # Автоматически распределяем по GPU
    trust_remote_code=True,
    torch_dtype=torch.bfloat16  # Тип данных для весов
)

print(f"  ✅ Модель загружена на: {model.device}")

# Подготавливаем модель для k-bit обучения
# Это включает gradient checkpointing и нормализует слои
model = prepare_model_for_kbit_training(model)

print("✅ Модель подготовлена к QLoRA обучению!")

---
## 🔧 Ячейка 5: Настройка LoRA адаптеров

In [ ]:
# Настраиваем LoRA адаптеры
# 
# LoRA (Low-Rank Adaptation) — добавляет обучаемые матрицы малого ранга
# к замороженным весам основной модели.
# Вместо обучения 7B параметров — обучаем ~10M параметров LoRA.

from peft import LoraConfig, TaskType

# Конфигурация LoRA
lora_config = LoraConfig(
    # r — ранг матриц LoRA (больше = больше параметров, лучше качество, но медленнее)
    # Для 7B модели r=16 — хороший баланс
    r=16,
    
    # alpha — коэффициент масштабирования LoRA (обычно = r или 2*r)
    lora_alpha=32,
    
    # target_modules — какие слои адаптировать
    # Для Mistral: q_proj, v_proj — минимум; добавляем k_proj, o_proj для лучшего качества
    target_modules=[
        "q_proj",   # Query проекция в self-attention
        "k_proj",   # Key проекция в self-attention
        "v_proj",   # Value проекция в self-attention
        "o_proj",   # Output проекция в self-attention
        "gate_proj", # Gate в MLP (SwiGLU активация)
        "up_proj",   # Up проекция в MLP
        "down_proj", # Down проекция в MLP
    ],
    
    # dropout для регуляризации LoRA слоёв
    lora_dropout=0.05,
    
    # bias — обычно не обучаем
    bias="none",
    
    # Тип задачи — языковое моделирование
    task_type=TaskType.CAUSAL_LM
)

# Применяем LoRA к модели
model = get_peft_model(model, lora_config)

# Выводим статистику параметров
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("📊 Статистика параметров модели:")
print(f"  🔧 Обучаемых параметров (LoRA): {trainable_params:,}")
print(f"  📦 Всего параметров: {total_params:,}")
print(f"  📉 Доля обучаемых: {100 * trainable_params / total_params:.2f}%")
print()
print("✅ LoRA адаптеры настроены!")
print(f"   Ранг (r): {lora_config.r}")
print(f"   Alpha: {lora_config.lora_alpha}")
print(f"   Целевые модули: {lora_config.target_modules}")

---
## 🏋️ Ячейка 6: Настройка и запуск обучения (SFTTrainer)

In [ ]:
# Настраиваем SFTTrainer и запускаем обучение
# 
# SFTTrainer (Supervised Fine-Tuning Trainer) из библиотеки TRL —
# специализированный тренер для instruction tuning LLM.

from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

# ========== ГИПЕРПАРАМЕТРЫ ОБУЧЕНИЯ ==========
# Настроены для GPU с 8-24GB VRAM
# При нехватке памяти: уменьши per_device_train_batch_size до 1

training_args = SFTConfig(
    # --- Основные параметры ---
    output_dir="./checkpoints",          # Папка для временных чекпоинтов
    num_train_epochs=3,                  # 3 эпохи — достаточно для 250 примеров
    
    # --- Размер батча ---
    per_device_train_batch_size=2,       # Батч на одно GPU
    per_device_eval_batch_size=2,        # Батч для валидации
    gradient_accumulation_steps=4,       # Накопление градиентов (эффективный батч = 2*4=8)
    
    # --- Оптимизатор ---
    optim="paged_adamw_32bit",           # paged_adamw — экономит память для 4-bit обучения
    learning_rate=2e-4,                  # Learning rate для LoRA (выше, чем для полного FT)
    weight_decay=0.001,                  # L2 регуляризация
    
    # --- Расписание LR ---
    lr_scheduler_type="cosine",          # Косинусное убывание LR
    warmup_ratio=0.03,                   # 3% шагов на разогрев
    
    # --- Логирование ---
    logging_steps=10,                    # Логировать loss каждые 10 шагов
    logging_dir="./logs",                # Папка для логов
    
    # --- Сохранение ---
    save_steps=50,                       # Сохранять чекпоинт каждые 50 шагов
    save_total_limit=2,                  # Хранить только 2 последних чекпоинта
    
    # --- Evaluation ---
    eval_strategy="steps",               # Оценивать каждые N шагов
    eval_steps=50,                       # Оценка каждые 50 шагов
    
    # --- Оптимизация памяти ---
    fp16=False,                          # НЕ fp16 (несовместимо с bfloat16)
    bf16=True,                           # Используем bfloat16 для A100/RTX30xx+
    gradient_checkpointing=True,         # Экономия памяти (чуть медленнее)
    
    # --- Прочее ---
    group_by_length=True,                # Группируем примеры по длине (быстрее)
    report_to="none",                    # Отключаем W&B (логируем через matplotlib)
    
    # --- Параметры SFT ---
    max_seq_length=512,                  # Максимальная длина последовательности
    dataset_text_field="text",           # Колонка с текстом в датасете
    packing=False,                       # Не упаковываем примеры (для ясности)
)

# Создаём SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print("✅ SFTTrainer настроен!")
print(f"\n📊 Параметры обучения:")
print(f"  Эпох: {training_args.num_train_epochs}")
print(f"  Батч на GPU: {training_args.per_device_train_batch_size}")
print(f"  Накопление градиентов: {training_args.gradient_accumulation_steps}")
print(f"  Эффективный батч: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Макс. длина: {training_args.max_seq_length} токенов")

In [ ]:
# Запускаем обучение!
# 
# Примерное время обучения (250 примеров, 3 эпохи):
# - RTX 3060 (12GB): ~45-60 минут
# - RTX 3090 (24GB): ~20-30 минут
# - T4 (Colab, 16GB): ~60-90 минут

import os
os.makedirs(ADAPTER_OUTPUT_DIR, exist_ok=True)

print("🚀 Начинаем обучение...")
print("   Прогресс будет показан ниже")
print()

# Запускаем обучение
train_result = trainer.train()

print("\n✅ Обучение завершено!")
print(f"  ⏱️ Время обучения: {train_result.metrics['train_runtime']:.1f} сек")
print(f"  📉 Финальный loss: {train_result.metrics['train_loss']:.4f}")
print(f"  🔢 Шагов выполнено: {train_result.metrics['train_steps_per_second']:.2f} шаг/сек")

---
## 💾 Ячейка 7: Сохранение LoRA адаптера

In [ ]:
# Сохраняем ТОЛЬКО LoRA адаптер (не всю модель!)
# 
# Размер адаптера: ~50-100MB (вместо 14GB для полной модели)
# При загрузке: базовая модель + адаптер = полноценная fine-tuned модель

print(f"💾 Сохраняем LoRA адаптер в '{ADAPTER_OUTPUT_DIR}'...")

# Сохраняем только адаптер (не полную модель)
trainer.model.save_pretrained(ADAPTER_OUTPUT_DIR)

# Сохраняем токенизатор (нужен для загрузки)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)

# Проверяем что сохранилось
import os
saved_files = os.listdir(ADAPTER_OUTPUT_DIR)

print(f"\n✅ Адаптер сохранён!")
print(f"📁 Файлы в '{ADAPTER_OUTPUT_DIR}':")
for f in saved_files:
    size = os.path.getsize(os.path.join(ADAPTER_OUTPUT_DIR, f)) / 1024**2
    print(f"  {f}: {size:.1f} MB")

print(f"\n📌 Примечание: полная модель НЕ сохранялась")
print(f"   Для загрузки используй: PeftModel.from_pretrained(base_model, '{ADAPTER_OUTPUT_DIR}')")

---
## 📈 Ячейка 8: График Training Loss

In [ ]:
# Строим график training loss
# Loss должен убывать — это значит модель обучается

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Извлекаем историю логов из тренера
logs = trainer.state.log_history

# Разделяем train loss и eval loss
train_steps = []
train_losses = []
eval_steps = []
eval_losses = []

for log in logs:
    if "loss" in log and "eval_loss" not in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])
    if "eval_loss" in log:
        eval_steps.append(log["step"])
        eval_losses.append(log["eval_loss"])

# Строим график
fig, ax = plt.subplots(figsize=(12, 5))

# Train loss
ax.plot(train_steps, train_losses, 
        color="#2196F3", linewidth=2, label="Train Loss", marker="o", markersize=4)

# Eval loss (если есть)
if eval_losses:
    ax.plot(eval_steps, eval_losses, 
            color="#F44336", linewidth=2, label="Eval Loss", 
            marker="s", markersize=6, linestyle="--")

# Оформление
ax.set_title("Training Loss — Mistral-7B QLoRA Fine-tuning", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Шаг (Step)", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_facecolor("#f8f9fa")
fig.patch.set_facecolor("white")

# Добавляем значения min/max
if train_losses:
    min_loss = min(train_losses)
    ax.axhline(y=min_loss, color="green", linestyle=":", alpha=0.5, label=f"Min loss: {min_loss:.4f}")
    ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n📊 Статистика обучения:")
if train_losses:
    print(f"  Начальный loss: {train_losses[0]:.4f}")
    print(f"  Финальный loss: {train_losses[-1]:.4f}")
    print(f"  Минимальный loss: {min(train_losses):.4f}")
    improvement = (train_losses[0] - train_losses[-1]) / train_losses[0] * 100
    print(f"  Улучшение: {improvement:.1f}%")
print("\n✅ График сохранён в 'training_loss.png'")

---
# 📊 ШАГ 3: ОЦЕНКА МОДЕЛИ

Сравниваем базовую Mistral vs Fine-tuned Mistral+LoRA на 20 примерах с помощью BERTScore.

## 🔄 Ячейка 9: Загрузка моделей для оценки

In [ ]:
# Освобождаем память перед загрузкой моделей для оценки
# (тренер больше не нужен)

import gc
import torch

print("🧹 Очищаем память после обучения...")

# Удаляем тренера и обученную модель из памяти GPU
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

# Показываем доступную VRAM
if torch.cuda.is_available():
    free_mem = torch.cuda.mem_get_info()[0] / 1024**3
    total_mem = torch.cuda.mem_get_info()[1] / 1024**3
    print(f"✅ VRAM: {free_mem:.1f}GB свободно из {total_mem:.1f}GB")

print("\n✅ Память очищена, готов к оценке!")

In [ ]:
# Загружаем БАЗОВУЮ модель (без LoRA) для сравнения
# 
# Базовая модель = оригинальный Mistral-7B-Instruct без дообучения

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("📦 Загружаем БАЗОВУЮ модель Mistral-7B...")

# Та же конфигурация 4-bit квантизации
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Для inference используем padding слева

# Загружаем базовую модель
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)
base_model.eval()  # Режим оценки (отключает dropout)

print("✅ Базовая модель загружена!")

In [ ]:
# Загружаем FINE-TUNED модель: базовая + LoRA адаптер

from peft import PeftModel

print(f"🔧 Загружаем LoRA адаптер из '{ADAPTER_OUTPUT_DIR}'...")

# Добавляем LoRA адаптер поверх базовой модели
# Сначала копируем базовую модель, потом накладываем адаптер
ft_model = PeftModel.from_pretrained(
    base_model,              # Базовая модель
    ADAPTER_OUTPUT_DIR,      # Путь к LoRA адаптеру
)
ft_model.eval()  # Режим оценки

print("✅ Fine-tuned модель загружена (базовая + LoRA адаптер)!")
print(f"\n📌 Для генерации используем:")
print(f"  base_model  → оригинальный Mistral-7B")
print(f"  ft_model    → Mistral-7B + LoRA ({ADAPTER_OUTPUT_DIR})")

---
## 🎯 Ячейка 10: Генерация ответов

In [ ]:
# Функция для генерации ответа от модели
# Используем inference-режим (torch.no_grad) для экономии памяти

import torch

def generate_response(model, tokenizer, instruction, max_new_tokens=256):
    """
    Генерирует ответ модели на инструкцию.
    
    Args:
        model: Модель (base или fine-tuned)
        tokenizer: Токенизатор
        instruction: Текст инструкции
        max_new_tokens: Максимум новых токенов в ответе
    
    Returns:
        str: Сгенерированный ответ
    """
    # Форматируем инструкцию в формат Mistral
    prompt = f"<s>[INST] {instruction} [/INST]"
    
    # Токенизируем
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)
    
    # Генерируем ответ
    with torch.no_grad():  # Отключаем вычисление градиентов
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,        # Низкая температура = более детерминированный ответ
            do_sample=True,         # Семплирование включено (нужно при temperature != 1)
            top_p=0.9,              # Nucleus sampling
            repetition_penalty=1.1, # Штраф за повторения
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Декодируем только новые токены (отрезаем prompt)
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return response.strip()


print("✅ Функция generate_response готова")

In [ ]:
# Выбираем 20 случайных примеров и генерируем ответы
# 
# Это самый долгий шаг оценки — примерно 5-15 минут

import random
from tqdm.notebook import tqdm

random.seed(RANDOM_SEED)

# Выбираем 20 случайных примеров из исходного датасета
eval_indices = random.sample(range(len(raw_data)), min(EVAL_SAMPLES, len(raw_data)))
eval_examples = [raw_data[i] for i in eval_indices]

print(f"📝 Выбрано {len(eval_examples)} примеров для оценки")
print("\n🔄 Генерируем ответы... (это займёт 5-15 минут)")

# Списки для хранения результатов
instructions = []
references = []
base_responses = []
ft_responses = []

# Проходим по каждому примеру
for i, example in enumerate(tqdm(eval_examples, desc="Оценка примеров")):
    instruction = example["instruction"]
    reference = example["response"]
    
    # Генерируем ответ базовой модели
    base_resp = generate_response(base_model, tokenizer, instruction)
    
    # Генерируем ответ fine-tuned модели
    ft_resp = generate_response(ft_model, tokenizer, instruction)
    
    # Сохраняем результаты
    instructions.append(instruction)
    references.append(reference)
    base_responses.append(base_resp)
    ft_responses.append(ft_resp)
    
    # Показываем прогресс каждые 5 примеров
    if (i + 1) % 5 == 0:
        print(f"\n  Пример {i+1}/{len(eval_examples)}:")
        print(f"  📋 Инструкция: {instruction[:80]}...")
        print(f"  ✅ Base: {base_resp[:80]}...")
        print(f"  🔧 Fine-tuned: {ft_resp[:80]}...")

print(f"\n✅ Генерация завершена! Обработано {len(instructions)} примеров")

---
## 📐 Ячейка 11: Вычисление BERTScore

In [ ]:
# Вычисляем BERTScore для базовой и fine-tuned модели
# 
# BERTScore сравнивает тексты через BERT эмбеддинги:
# - Precision: насколько слова предсказанного ответа совпадают с reference
# - Recall: насколько слова reference покрыты предсказанным ответом  
# - F1: гармоническое среднее Precision и Recall
# 
# Диапазон: от -1 до 1 (выше = лучше)
# Обычно хорошие значения: F1 > 0.85

from bert_score import score as bert_score

print("📐 Вычисляем BERTScore...")
print("   (первый запуск загрузит модель deberta-xlarge-mnli ~800MB)\n")

# BERTScore для БАЗОВОЙ модели
print("1️⃣ Оцениваем базовую модель...")
base_P, base_R, base_F1 = bert_score(
    cands=base_responses,    # Кандидаты (ответы базовой модели)
    refs=references,          # Эталонные ответы
    lang="en",               # Язык (измени на "ru" если датасет на русском)
    verbose=True,
    batch_size=8             # Размер батча для BERTScore
)

# BERTScore для FINE-TUNED модели
print("\n2️⃣ Оцениваем fine-tuned модель...")
ft_P, ft_R, ft_F1 = bert_score(
    cands=ft_responses,      # Кандидаты (ответы fine-tuned модели)
    refs=references,          # Те же эталонные ответы
    lang="en",               # Язык
    verbose=True,
    batch_size=8
)

# Конвертируем тензоры в numpy для удобной работы
base_P = base_P.numpy()
base_R = base_R.numpy()
base_F1 = base_F1.numpy()
ft_P = ft_P.numpy()
ft_R = ft_R.numpy()
ft_F1 = ft_F1.numpy()

print("\n" + "=" * 50)
print("📊 СРЕДНИЕ ЗНАЧЕНИЯ BERTScore")
print("=" * 50)
print(f"\n🔵 Базовая модель (Mistral-7B-Instruct):")
print(f"   Precision: {base_P.mean():.4f}")
print(f"   Recall:    {base_R.mean():.4f}")
print(f"   F1:        {base_F1.mean():.4f}")
print(f"\n🟢 Fine-tuned модель (Mistral-7B + LoRA):")
print(f"   Precision: {ft_P.mean():.4f}")
print(f"   Recall:    {ft_R.mean():.4f}")
print(f"   F1:        {ft_F1.mean():.4f}")
print()

# Считаем улучшение
f1_improvement = (ft_F1.mean() - base_F1.mean()) / abs(base_F1.mean()) * 100
print(f"📈 Изменение F1: {'+' if f1_improvement >= 0 else ''}{f1_improvement:.2f}%")

---
## 📋 Ячейка 12: Сравнительная таблица результатов

In [ ]:
# Создаём и сохраняем сравнительную таблицу результатов

import pandas as pd

# Создаём DataFrame с результатами
results_df = pd.DataFrame({
    "#": range(1, len(instructions) + 1),
    "Instruction": instructions,
    "Reference": references,
    "Base Response": base_responses,
    "Fine-tuned Response": ft_responses,
    "Base BERTScore F1": base_F1.round(4),
    "FT BERTScore F1": ft_F1.round(4),
    "Delta F1": (ft_F1 - base_F1).round(4)  # Положительное = FT лучше
})

# Сохраняем в CSV
csv_path = "evaluation_results.csv"
results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")  # utf-8-sig для корректного отображения в Excel

print(f"✅ Таблица сохранена в '{csv_path}'")
print(f"\n📊 Сводная таблица (первые 5 примеров):")
print()

# Отображаем таблицу (усечённые тексты для читаемости)
display_df = results_df[["#", "Instruction", "Base BERTScore F1", "FT BERTScore F1", "Delta F1"]].copy()
display_df["Instruction"] = display_df["Instruction"].str[:60] + "..."

# Цветовая подсветка: зелёный = FT лучше, красный = хуже
def highlight_delta(val):
    if val > 0:
        return "background-color: #c8e6c9"  # Зелёный
    elif val < 0:
        return "background-color: #ffcdd2"  # Красный
    return ""

styled_df = display_df.style.applymap(highlight_delta, subset=["Delta F1"])
display(styled_df.head(5))

print(f"\n📈 Примеров где FT лучше: {(results_df['Delta F1'] > 0).sum()}/{len(results_df)}")
print(f"📉 Примеров где FT хуже: {(results_df['Delta F1'] < 0).sum()}/{len(results_df)}")
print(f"➡️ Примеров без изменений: {(results_df['Delta F1'] == 0).sum()}/{len(results_df)}")

---
## 📊 Ячейка 13: Итоговый график BERTScore

In [ ]:
# Строим итоговый сравнительный график BERTScore

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    "BERTScore: Базовая Mistral-7B vs Fine-tuned Mistral-7B+LoRA",
    fontsize=15, fontweight="bold", y=1.02
)

metrics = [
    ("Precision", base_P, ft_P),
    ("Recall", base_R, ft_R),
    ("F1", base_F1, ft_F1),
]

x = np.arange(len(instructions))
width = 0.35  # Ширина столбца

for ax, (metric_name, base_vals, ft_vals) in zip(axes, metrics):
    # Столбчатая диаграмма
    bars1 = ax.bar(x - width/2, base_vals, width, 
                   label="Base", color="#90CAF9", alpha=0.85, edgecolor="#1565C0")
    bars2 = ax.bar(x + width/2, ft_vals, width, 
                   label="Fine-tuned", color="#A5D6A7", alpha=0.85, edgecolor="#2E7D32")
    
    # Горизонтальные линии средних значений
    ax.axhline(y=base_vals.mean(), color="#1565C0", linestyle="--", 
               alpha=0.7, linewidth=1.5, label=f"Base avg: {base_vals.mean():.4f}")
    ax.axhline(y=ft_vals.mean(), color="#2E7D32", linestyle="--", 
               alpha=0.7, linewidth=1.5, label=f"FT avg: {ft_vals.mean():.4f}")
    
    # Настройки осей
    ax.set_title(f"BERTScore {metric_name}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Пример №", fontsize=11)
    ax.set_ylabel(f"BERTScore {metric_name}", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([str(i+1) for i in x], fontsize=8)
    ax.legend(fontsize=9, loc="lower right")
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_facecolor("#f8f9fa")
    
    # Устанавливаем разумный диапазон оси Y
    all_vals = np.concatenate([base_vals, ft_vals])
    y_min = max(0, all_vals.min() - 0.05)
    y_max = min(1, all_vals.max() + 0.05)
    ax.set_ylim(y_min, y_max)

fig.patch.set_facecolor("white")
plt.tight_layout()
plt.savefig("bertscore_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ График сохранён в 'bertscore_comparison.png'")

In [ ]:
# Дополнительный график: boxplot для распределения F1 score

fig, ax = plt.subplots(figsize=(8, 6))

data_to_plot = [base_F1, ft_F1]
labels = ["Base\nMistral-7B", "Fine-tuned\nMistral-7B+LoRA"]
colors = ["#90CAF9", "#A5D6A7"]

bp = ax.boxplot(
    data_to_plot,
    labels=labels,
    patch_artist=True,
    showmeans=True,
    meanprops={"marker": "D", "markerfacecolor": "red", "markeredgecolor": "darkred", "markersize": 8}
)

# Закрашиваем боксы
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_title("Распределение BERTScore F1\n(◆ = среднее значение)", 
             fontsize=13, fontweight="bold")
ax.set_ylabel("BERTScore F1", fontsize=12)
ax.grid(True, alpha=0.3, axis="y")
ax.set_facecolor("#f8f9fa")
fig.patch.set_facecolor("white")

# Добавляем аннотации со средними значениями
for i, (vals, label) in enumerate(zip(data_to_plot, labels), 1):
    ax.annotate(
        f"μ={vals.mean():.4f}",
        xy=(i, vals.mean()),
        xytext=(i + 0.15, vals.mean()),
        fontsize=10,
        color="darkred",
        fontweight="bold"
    )

plt.tight_layout()
plt.savefig("bertscore_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Boxplot сохранён в 'bertscore_boxplot.png'")

---
## 📝 Ячейка 14: Итоговые выводы

In [ ]:
# Автоматически генерируем текстовый вывод на основе результатов

import numpy as np

# Вычисляем итоговые метрики
avg_base_f1 = base_F1.mean()
avg_ft_f1 = ft_F1.mean()
f1_delta = avg_ft_f1 - avg_base_f1
f1_delta_pct = f1_delta / abs(avg_base_f1) * 100

better_count = (ft_F1 > base_F1).sum()
worse_count = (ft_F1 < base_F1).sum()
equal_count = (ft_F1 == base_F1).sum()

print("=" * 60)
print("📊 ИТОГОВЫЕ РЕЗУЛЬТАТЫ ОЦЕНКИ")
print("=" * 60)
print()
print(f"{'Метрика':<25} {'Базовая':>12} {'Fine-tuned':>12} {'Разница':>12}")
print("-" * 62)
print(f"{'BERTScore Precision':<25} {base_P.mean():>12.4f} {ft_P.mean():>12.4f} {ft_P.mean()-base_P.mean():>+12.4f}")
print(f"{'BERTScore Recall':<25} {base_R.mean():>12.4f} {ft_R.mean():>12.4f} {ft_R.mean()-base_R.mean():>+12.4f}")
print(f"{'BERTScore F1':<25} {avg_base_f1:>12.4f} {avg_ft_f1:>12.4f} {f1_delta:>+12.4f}")
print("-" * 62)
print()
print(f"Примеров где FT лучше:    {better_count}/{len(ft_F1)}")
print(f"Примеров где FT хуже:     {worse_count}/{len(ft_F1)}")
print(f"Примеров без изменений:   {equal_count}/{len(ft_F1)}")
print()
print("=" * 60)
print("💬 ВЫВОД")
print("=" * 60)

# Автоматически генерируем вывод
if f1_delta > 0.01:
    print(f"""
✅ Fine-tuning УЛУЧШИЛ модель.

BERTScore F1 вырос с {avg_base_f1:.4f} до {avg_ft_f1:.4f}
(улучшение на {f1_delta_pct:.2f}%).

Из {len(ft_F1)} тестовых примеров fine-tuned модель показала лучшие 
результаты в {better_count} случаях ({better_count/len(ft_F1)*100:.0f}%).

Это говорит о том, что модель успешно адаптировалась к формату
и стилю ответов из обучающего датасета.
""")
elif f1_delta > -0.01:
    print(f"""
➡️ Fine-tuning оказал НЕЗНАЧИТЕЛЬНОЕ влияние на модель.

BERTScore F1 изменился с {avg_base_f1:.4f} до {avg_ft_f1:.4f}
(изменение всего {f1_delta_pct:+.2f}%).

Возможные причины:
1. Датасет слишком мал (250 примеров) — рекомендуется 1000+
2. Базовая модель уже хорошо справляется с задачей
3. Нужно больше эпох обучения (попробуй 5-10 эпох)
4. BERTScore может не улавливать специфику задачи

Рекомендация: провести ручной анализ ответов моделей.
""")
else:
    print(f"""
⚠️ Fine-tuning УХУДШИЛ модель по метрике BERTScore.

BERTScore F1 снизился с {avg_base_f1:.4f} до {avg_ft_f1:.4f}
(ухудшение на {abs(f1_delta_pct):.2f}%).

Возможные причины:
1. Catastrophic forgetting — модель «забыла» общие знания
2. Слишком высокий learning rate ({2e-4}) — попробуй уменьшить до 1e-4
3. Слишком много эпох — попробуй 1-2 эпохи
4. Датасет содержит некачественные примеры (проверь cleaned_dataset.jsonl)
5. BERTScore не подходит для твоей задачи

Рекомендация: провести ручной анализ ответов и качества датасета.
""")

print("=" * 60)
print("\n📁 Созданные файлы:")
print(f"  ./adapter/                 — LoRA адаптер")
print(f"  training_loss.png          — График обучения")
print(f"  evaluation_results.csv     — Таблица сравнения")
print(f"  bertscore_comparison.png   — График BERTScore")
print(f"  bertscore_boxplot.png      — Boxplot распределения")